# IRF peak and FWHM

Point `DATA_DIR` at a folder that holds a histogram export from the TDC software (`*_Histogram 0.csv`).
For every CSV in that folder the notebook:

1. reads the histogram and keeps one laser period (0 to `PERIOD_NS`),
2. finds the peak and the flat background,
3. measures the FWHM at half way between background and peak, on a light rebin (about 9.4 ps) so the bin-to-bin TDC ripple does not move the crossings,
4. plots the raw histogram with the FWHM drawn on it, plus the full period on a log scale as an inset,
5. if `SAVE` is True, saves the figure next to the CSV as `<csv name>_fwhm.png`.


## 1. Settings

In [ ]:
from pathlib import Path

# Folder with the CSV export. Relative paths are taken from the repository root.
DATA_DIR = r"irf-library\test-irf\261104-IRF-blackplastic-50p-10minwarmup"
CSV_PATTERN = "*.csv"  # every CSV in the folder is processed

PERIOD_NS = 12.5  # laser period: 80 MHz
REBIN_PS = 9.4  # target width of the rebin used for the crossings (0 = no rebin)
VIEW_NS = (-0.7, 1.1)  # x range of the main plot, relative to the peak
TITLE = ""  # empty: the CSV file name is used
SAVE = True  # save the figure next to the CSV

## 2. Functions

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def repo_root(start: Path | None = None) -> Path:
    """First parent folder that contains irf-library (falls back to the current folder)."""
    start = start or Path.cwd()
    for p in (start, *start.parents):
        if (p / "irf-library").is_dir():
            return p
    return start


def load_histogram(path: Path, period_ns: float = PERIOD_NS):
    """Bin centres [ns], counts and bin width [ns], restricted to one laser period."""
    a = np.loadtxt(path, delimiter=",", comments="#", ndmin=2)
    t = a[:, 0] * 1e9  # left edges in s -> ns
    c = a[:, 1]
    bw = (t[-1] - t[0]) / (t.size - 1)  # the CSV prints 6 digits only, so not np.diff
    keep = (t > -1e-6) & (t < period_ns - 1e-6)
    return t[keep] + bw / 2, c[keep], bw


def _cross(x, y, level, rising):
    """First rising or last falling crossing of `level`, linear interpolation between bins."""
    idx = np.where(y >= level)[0]
    if idx.size == 0:
        return np.nan
    j = idx[0] if rising else idx[-1]
    if rising:
        if j == 0:
            return x[0]
        return x[j - 1] + (level - y[j - 1]) * (x[j] - x[j - 1]) / (y[j] - y[j - 1])
    if j == y.size - 1:
        return x[-1]
    return x[j] + (y[j] - level) * (x[j + 1] - x[j]) / (y[j] - y[j + 1])


def analyse(t, c, bw, rebin_ps: float = REBIN_PS):
    """Peak, background and FWHM. The crossings are read on a rebin of about `rebin_ps`."""
    k = max(1, int(round(rebin_ps / (bw * 1e3)))) if rebin_ps > 0 else 1
    n = (t.size // k) * k
    tr = t[:n].reshape(-1, k).mean(1)
    cr = c[:n].reshape(-1, k).mean(1)  # mean, so it stays in counts per raw bin
    ip = int(np.argmax(cr))
    tp, peak = float(tr[ip]), float(cr[ip])
    period = t[-1] + bw / 2
    flat = ((t > 1.0) & (t < tp - 1.5)) | ((t > tp + 3.5) & (t < period - 0.2))
    bg = float(np.median(c[flat])) if flat.any() else 0.0
    half = bg + 0.5 * (peak - bg)
    t_l = _cross(tr[: ip + 1], cr[: ip + 1], half, rising=True)
    t_r = _cross(tr[ip:], cr[ip:], half, rising=False)
    return dict(
        tr=tr, cr=cr, k=k, peak_ns=tp, peak=peak, bg=bg, half=half, t_left=t_l, t_right=t_r, fwhm_ps=(t_r - t_l) * 1e3
    )


def plot_fwhm(t, c, bw, r, title, view_ns=VIEW_NS):
    tp, peak, half = r["peak_ns"], r["peak"], r["half"]
    lo, hi = tp + view_ns[0], tp + view_ns[1]
    fig, ax = plt.subplots(figsize=(11, 6.5))

    m = (t > lo) & (t < hi)
    ax.step(t[m], c[m], where="mid", lw=0.6, color="0.45", label=f"raw data, {bw * 1e3:.2f} ps bins")
    if r["k"] > 1:
        mr = (r["tr"] > lo) & (r["tr"] < hi)
        ax.plot(
            r["tr"][mr],
            r["cr"][mr],
            lw=1.4,
            color="tab:blue",
            label=f"{r['k'] * bw * 1e3:.1f} ps rebin (used for the crossings)",
        )

    ax.axhline(peak, color="tab:blue", lw=0.8, ls=":", alpha=0.6)
    ax.axhline(half, color="tab:red", lw=0.8, ls="--", alpha=0.6)
    ax.vlines([r["t_left"], r["t_right"]], 0, half, color="tab:red", lw=0.8, ls="--", alpha=0.6)
    ax.annotate(
        "",
        xy=(r["t_right"], half),
        xytext=(r["t_left"], half),
        arrowprops=dict(arrowstyle="<->", color="tab:red", lw=2),
    )
    ax.text(
        (r["t_left"] + r["t_right"]) / 2,
        half,
        f"FWHM = {r['fwhm_ps']:.0f} ps",
        color="tab:red",
        ha="center",
        va="bottom",
        fontsize=14,
        fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="tab:red", alpha=0.9),
    )
    ax.text(tp + 0.02, peak, f"peak {peak:,.0f} counts/bin at {tp:.2f} ns", fontsize=8, color="tab:blue", va="bottom")

    ax.set_xlim(lo, hi)
    ax.set_ylim(0, max(peak, c[m].max()) * 1.12)
    ax.set_xlabel("time after SYNC [ns]")
    ax.set_ylabel(f"counts per {bw * 1e3:.2f} ps bin")
    ax.set_title(title)
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(alpha=0.3)

    ins = ax.inset_axes([0.66, 0.38, 0.32, 0.28])
    ins.step(t, np.clip(c, 1, None), where="mid", lw=0.4, color="0.45")
    ins.set_yscale("log")
    ins.set_xlim(0, t[-1] + bw / 2)
    ins.set_ylim(max(1, 0.5 * np.percentile(c[c > 0], 5)) if (c > 0).any() else 1, peak * 3)
    ins.set_title("full period, log", fontsize=8)
    ins.tick_params(labelsize=7)
    ins.grid(alpha=0.3)
    ins.axvspan(lo, hi, color="tab:red", alpha=0.08)

    fig.tight_layout()
    return fig

## 3. Find the CSV, measure and plot

In [ ]:
data_dir = Path(DATA_DIR)
if not data_dir.is_absolute():
    data_dir = repo_root() / data_dir
csvs = sorted(data_dir.glob(CSV_PATTERN))
if not csvs:
    raise FileNotFoundError(f"no file matching {CSV_PATTERN} in {data_dir}")

results = {}
for csv in csvs:
    t, c, bw = load_histogram(csv)
    r = analyse(t, c, bw)
    results[csv.name] = r
    print(
        f"{csv.name}\n"
        f"  {c.size} bins of {bw * 1e3:.3f} ps, {c.sum() / 1e6:.2f} M counts in one period\n"
        f"  peak {r['peak']:,.0f} counts/bin at {r['peak_ns']:.3f} ns, background {r['bg']:.0f} counts/bin\n"
        f"  FWHM {r['fwhm_ps']:.1f} ps (crossings {r['t_left']:.4f} and {r['t_right']:.4f} ns)"
    )
    fig = plot_fwhm(t, c, bw, r, TITLE or csv.stem)
    if SAVE:
        out = csv.with_name(csv.stem + "_fwhm.png")
        fig.savefig(out, dpi=110)
        print(f"  saved {out}")
    plt.show()